### Step 1: load the features

In [1]:
import numpy as np
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="supply-chain-analytics-510105")
df = client.query(
    "select * from `supply-chain-analytics-510105.dbt_dev_ml.ml_late_delivery_features`"
).to_dataframe()

df["purchased_at"] = pd.to_datetime(df["purchased_at"], utc=True).dt.tz_convert(None)
df["is_late"] = df["is_late"].astype(int)
print(df.shape, f"late rate: {df['is_late'].mean():.1%}")

/home/justin/supply-chain-analytics/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(96189, 25) late rate: 6.8%


### Step 2: time-based split

In [2]:
month = df["purchased_at"].dt.to_period("M")

train = df[month <= "2017-12"]
valid = df[(month >= "2018-01") & (month <= "2018-03")]
test  = df[(month >= "2018-04") & (month <= "2018-07")]

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:<6} {len(d):>6,} orders   late rate {d['is_late'].mean():.1%}")

train  43,412 orders   late rate 5.7%
valid  20,627 orders   late rate 12.9%
test   25,799 orders   late rate 4.0%


### Step 3: features

In [13]:
num_cols = [
    "promised_days", "approval_hours", "seller_deadline_slack_days",
     "purchase_dow", "max_distance_km",
    "n_items", "n_sellers", "items_value", "freight_value", "freight_ratio",
    "total_weight_g", "total_volume_cm3",
    "seller_prior_orders", "seller_prior_late_rate", "seller_prior_missed_rate",
]
bool_cols = ["is_peak_season", "is_same_state"]
cat_cols  = ["customer_state", "seller_state", "main_category"]

# the same category list everywhere, so train and test are encoded identically
categories = {c: sorted(df[c].fillna("unknown").unique()) for c in cat_cols}

def prep(d):
    X = d[num_cols + bool_cols + cat_cols].copy()
    X[num_cols]  = X[num_cols].astype(float)
    X[bool_cols] = X[bool_cols].astype(float)
    for c in cat_cols:
        X[c] = pd.Categorical(X[c].fillna("unknown"), categories=categories[c])
    return X, d["is_late"].to_numpy()

X_train, y_train = prep(train)
X_valid, y_valid = prep(valid)
X_test,  y_test  = prep(test)

### Step 4: the evaluation function

In [14]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(name, y, p, top=0.10):
    """ROC AUC, PR AUC, and what happens if customer service contacts the riskiest 10%."""
    k = int(len(p) * top)
    flagged = np.argsort(-p)[:k]
    return {
        "model": name,
        "roc_auc": round(roc_auc_score(y, p), 3),
        "pr_auc": round(average_precision_score(y, p), 3),
        "precision_top10%": round(y[flagged].mean(), 3),
        "recall_top10%": round(y[flagged].sum() / y.sum(), 3),
    }

### Step 5: a baseline and two models

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

results = []

# Baseline: a simple rule, "the seller's past late rate" (overall rate for new sellers)
fallback = y_train.mean()
base = X_valid["seller_prior_late_rate"].fillna(fallback).to_numpy()
results.append(evaluate("baseline: seller history", y_valid, base))

# Logistic regression
logreg = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols + bool_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat_cols),
    ]),
    LogisticRegression(max_iter=2000),
)
logreg.fit(X_train, y_train)
results.append(evaluate("logistic regression", y_valid, logreg.predict_proba(X_valid)[:, 1]))

# Gradient boosting (handles missing values and categories natively)
gbm = HistGradientBoostingClassifier(
    categorical_features="from_dtype",
    learning_rate=0.05, max_iter=150, max_leaf_nodes=15,
    min_samples_leaf=100, l2_regularization=1.0, random_state=42,
)
gbm.fit(X_train, y_train)
results.append(evaluate("gradient boosting", y_valid, gbm.predict_proba(X_valid)[:, 1]))

pd.DataFrame(results)

,model,roc_auc,pr_auc,precision_top10%,recall_top10%
0,baseline: seller history,0.548,0.145,0.148,0.115
1,logistic regression,0.715,0.285,0.354,0.274
2,gradient boosting,0.662,0.213,0.247,0.192


### Step 6: final model on train + validation, one test

In [19]:
from sklearn.base import clone

# retrain on everything before the test period (Jan 2017 - Mar 2018)
X_trval = pd.concat([X_train, X_valid])
y_trval = np.concatenate([y_train, y_valid])

final = clone(logreg).fit(X_trval, y_trval)
p_test = final.predict_proba(X_test)[:, 1]

base_test = X_test["seller_prior_late_rate"].fillna(y_trval.mean()).to_numpy()

test_results = pd.DataFrame([
    evaluate("baseline: seller history", y_test, base_test),
    evaluate("logistic regression (final)", y_test, p_test),
])
print(f"test late rate (random PR AUC): {y_test.mean():.3f}")
test_results

test late rate (random PR AUC): 0.040


,model,roc_auc,pr_auc,precision_top10%,recall_top10%
0,baseline: seller history,0.548,0.048,0.068,0.17
1,logistic regression (final),0.734,0.106,0.128,0.32


### Step 7: which features matter?

In [20]:
from sklearn.inspection import permutation_importance

r = permutation_importance(
    final, X_test, y_test,
    scoring="average_precision", n_repeats=5, random_state=42,
)
importance = (
    pd.Series(r.importances_mean, index=X_test.columns)
    .sort_values(ascending=False)
    .round(4)
)
importance.head(10)

promised_days               0.0431
max_distance_km             0.0304
is_same_state               0.0200
customer_state              0.0150
seller_state                0.0119
seller_prior_missed_rate    0.0056
n_items                     0.0038
seller_prior_late_rate      0.0020
n_sellers                   0.0019
total_volume_cm3            0.0008
dtype: float64

### Step 8: save the model and results

In [22]:
import joblib
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

joblib.dump(final, out / "late_delivery_logreg.joblib")
test_results.to_csv(out / "late_delivery_test_results.csv", index=False)
importance.to_csv(out / "late_delivery_feature_importance.csv", header=["pr_auc_drop"])
print("saved:", sorted(p.name for p in out.iterdir()))

saved: ['late_delivery_feature_importance.csv', 'late_delivery_logreg.joblib', 'late_delivery_test_results.csv']
